In order to increase the amount of fake news, we will do external scraping by getting news from another sources.And we will combine it with our scraped.csv from data manipulation and add it to visualization part.

In [72]:
!pip install newspaper3k tqdm
!pip install lxml_html_clean

In [73]:
import pandas as pd
import newspaper
from newspaper import Article
from tqdm import tqdm
import nltk
import os

In [74]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [75]:
current_filename = "scraped_news.csv"
output_filename = "scraped_news_augmented.csv"

In [76]:
try:  # loop for printing current file data
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())

        print(f"Current file: {current_filename}")
        print(f"Number of rows: {len(df_main)}")
        print(f"Number of fake rows: {len(df_main[df_main['label']==1])}")
    else:
        print(f"'{current_filename}' not found.")
        exit()
except Exception as e:
    print(f"Reading error: {e}")
    exit()


Current file: scraped_news.csv
Number of rows: 11125
Number of fake rows: 140


In [77]:
targets = [
    {"url": "https://www.theonion.com/", "label": 1, "source": "The Onion"},
    {"url": "https://babylonbee.com/", "label": 1, "source": "The Babylon Bee"},
    {"url": "https://www.duffelblog.com/", "label": 1, "source": "Duffel Blog"},
    {"url": "https://reductress.com/", "label": 1, "source": "Reductress"}
]  # we get the fake news from these targets


we only get fake news from target sources.

In [78]:
new_fake_data = []  # empty list for new articles

for target in targets:  # loop for scanning article sources
    print(f"'{target['source']}' scanning...")

    try:

        paper = newspaper.build(target['url'], memoize_articles=False)
        print(f"Number of articles found: {len(paper.articles)}")

        limit = 600 # get 600 links from each
        count = 0
        skipped = 0

        for article in tqdm(paper.articles[:limit]):  # if exists in current dataset
            if article.url in existing_urls:
                skipped += 1
                continue

            try:
                article.download()
                article.parse()


                if len(article.text) > 200:
                    new_fake_data.append({
                        "title": article.title,
                        "news_url": article.url,
                        "source": target['source'],
                        "label": target['label'],
                        "full_text": article.text
                    })

                    existing_urls.add(article.url)
                    count += 1
            except:
                continue

        print(f"Number of added articles: {count} (We had: {skipped})")

    except Exception as e:
        print(f"Error: ({target['source']}): {e}")



'The Onion' scanning...


CRITICAL:newspaper.network:[REQUEST FAILED] 403 Client Error: Forbidden for url: https://membership.theonion.com/
CRITICAL:newspaper.network:[REQUEST FAILED] 403 Client Error: Forbidden for url: https://membership.theonion.com/
CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://theonion.com/feeds


Number of articles found: 152


100%|██████████| 152/152 [00:49<00:00,  3.08it/s]


Number of added articles: 105 (We had: 0)
'The Babylon Bee' scanning...


...

Number of added articles: 15 (We had: 0)
'Reductress' scanning...


CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://reductress.com/rss/1/
CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://reductress.com/feeds/


Number of articles found: 230


100%|██████████| 230/230 [02:28<00:00,  1.55it/s]

Number of added articles: 182 (We had: 0)


In [79]:
if len(new_fake_data) > 0:
    df_new_fake = pd.DataFrame(new_fake_data)


    df_augmented = pd.concat([df_main, df_new_fake], ignore_index=True)

    df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nSCRAPING AND COMBINING COMPLETED")
    print(f"Num of articles before: {len(df_main)}")
    print(f"Num of added fake articles: {len(df_new_fake)}")
    print(f"Current articles: {len(df_augmented)}")
    print(f"Distributions:\n{df_augmented['label'].value_counts()}")

    df_augmented.to_csv(output_filename, index=False)
    print(f"File saved: '{output_filename}'")

    df = df_augmented

else:
    print("\nNothing found.")


SCRAPING AND COMBINING COMPLETED
Num of articles before: 11125
Num of added fake articles: 338
Current articles: 11463
Distributions:
label
0    10985
1      478
Name: count, dtype: int64
File saved: 'scraped_news_augmented.csv'


In order to increase fake news, we will use another sources.

In [80]:
current_filename = "scraped_news_augmented.csv"
output_filename = "scraped_news_augmented.csv"

In [81]:
try:  # loop for printing current file data
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())

        print(f"Current file: {current_filename}")
        print(f"Number of rows: {len(df_main)}")
        print(f"Number of fake rows: {len(df_main[df_main['label']==1])}")
    else:
        print(f"'{current_filename}' not found.")
        df_main = pd.DataFrame()
        existing_urls = set()
except Exception as e:
    print(f"Reading error: {e}")
    exit()

Current file: scraped_news_augmented.csv
Number of rows: 11463
Number of fake rows: 478


In [82]:
targets = [
    {"url": "https://www.thedailymash.co.uk/", "label": 1, "source": "The Daily Mash"},
    {"url": "https://www.betootaadvocate.com/", "label": 1, "source": "Betoota Advocate"},
    {"url": "https://chaser.com.au/", "label": 1, "source": "The Chaser"},
    {"url": "https://burrardstreetjournal.com/", "label": 1, "source": "Burrard Street Journal"},
    {"url": "https://www.derfmagazine.com/", "label": 1, "source": "Derf Magazine"}
] # we get the fake news from these targets

In [83]:
new_fake_data = []

In [84]:
for target in targets:  # loop for scanning article sources
    print(f"'{target['source']}' scanning...")

    try:
        paper = newspaper.build(target['url'], memoize_articles=False)
        print(f"Number of articles found: {len(paper.articles)}")

        limit = 600 # get 600 links from each
        count = 0
        skipped = 0

        for article in tqdm(paper.articles[:limit]):  # if exists in current dataset
            if article.url in existing_urls:
                skipped += 1
                continue

            try:
                article.download()
                article.parse()

                if len(article.text) > 200:
                    new_fake_data.append({
                        "title": article.title,
                        "news_url": article.url,
                        "source": target['source'],
                        "label": target['label'],
                        "full_text": article.text
                    })

                    existing_urls.add(article.url)
                    count += 1
            except:
                continue

        print(f"Number of added articles: {count} (We had: {skipped})")

    except Exception as e:
        print(f"Error: ({target['source']}): {e}")

'The Daily Mash' scanning...
Number of articles found: 112


100%|██████████| 112/112 [00:56<00:00,  1.97it/s]


Number of added articles: 92 (We had: 0)
'Betoota Advocate' scanning...


CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://www.betootaadvocate.com/rss/
CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://www.betootaadvocate.com/feed/
CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://www.betootaadvocate.com/feeds/


Number of articles found: 28


...

Number of added articles: 0 (We had: 0)
'Derf Magazine' scanning...


CRITICAL:newspaper.extractors:Must extract urls from either html, text or doc!
CRITICAL:newspaper.network:[REQUEST FAILED] 500 Server Error: Internal Server Error for url: https://www.derfmagazine.com/
CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://www.derfmagazine.com/feed
CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://www.derfmagazine.com/rss
CRITICAL:newspaper.network:[REQUEST FAILED] 500 Server Error: Internal Server Error for url: https://www.derfmagazine.com/feeds/


Number of articles found: 0


0it [00:00, ?it/s]

Number of added articles: 0 (We had: 0)


In [85]:
if len(new_fake_data) > 0:
    df_new = pd.DataFrame(new_fake_data)

    df_augmented = pd.concat([df_main, df_new], ignore_index=True)

    df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nSCRAPING AND COMBINING COMPLETED")
    print(f"Num of articles before: {len(df_main)}")
    print(f"Num of added fake articles: {len(df_new)}")
    print(f"Current articles: {len(df_augmented)}")
    print(f"Distributions:\n{df_augmented['label'].value_counts()}")

    df_augmented.to_csv(output_filename, index=False)
    print(f"File saved: '{output_filename}'")

    df = df_augmented

else:
    print("\nNothing found.")


SCRAPING AND COMBINING COMPLETED
Num of articles before: 11463
Num of added fake articles: 133
Current articles: 11596
Distributions:
label
0    10985
1      611
Name: count, dtype: int64
File saved: 'scraped_news_augmented.csv'


We have to try another round because its not enough.

In [86]:
current_filename = "scraped_news_augmented.csv"
output_filename = "scraped_news_augmented.csv"

try:
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())

        print(f"Current file: {current_filename}")
        print(f"Number of rows: {len(df_main)}")
        print(f"Number of fake rows: {len(df_main[df_main['label']==1])}")
    else:
        print(f"'{current_filename}' not found.")
        exit()
except Exception as e:
    print(f"Reading error: {e}")
    exit()

targets = [
    {"url": "https://thehardtimes.net/", "label": 1, "source": "The Hard Times"},
    {"url": "https://theshovel.com.au/", "label": 1, "source": "The Shovel"},
    {"url": "https://newsthump.com/", "label": 1, "source": "NewsThump"},
    {"url": "https://weeklyworldnews.com/", "label": 1, "source": "Weekly World News"},
    {"url": "https://www.burrardstreetjournal.com/", "label": 1, "source": "Burrard Street Journal"}
]

new_fake_data = []

for target in targets:
    print(f"'{target['source']}' scanning...")

    try:
        paper = newspaper.build(target['url'], memoize_articles=False)
        print(f"Number of articles found: {len(paper.articles)}")

        limit = 800
        count = 0
        skipped = 0

        for article in tqdm(paper.articles[:limit]):
            if article.url in existing_urls:
                skipped += 1
                continue

            try:
                article.download()
                article.parse()

                if len(article.text) > 150:
                    new_fake_data.append({
                        "title": article.title,
                        "news_url": article.url,
                        "source": target['source'],
                        "label": target['label'],
                        "full_text": article.text
                    })

                    existing_urls.add(article.url)
                    count += 1
            except:
                continue

        print(f"Number of added articles: {count} (We had: {skipped})")

    except Exception as e:
        print(f"Error: ({target['source']}): {e}")

if len(new_fake_data) > 0:
    df_new = pd.DataFrame(new_fake_data)

    df_augmented = pd.concat([df_main, df_new], ignore_index=True)

    df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nSCRAPING AND COMBINING COMPLETED")
    print(f"Num of articles before: {len(df_main)}")
    print(f"Num of added fake articles: {len(df_new)}")
    print(f"Current articles: {len(df_augmented)}")
    print(f"Distributions:\n{df_augmented['label'].value_counts()}")

    df_augmented.to_csv(output_filename, index=False)
    print(f"File saved: '{output_filename}'")

    df = df_augmented

else:
    print("\nNothing found.")

Current file: scraped_news_augmented.csv
Number of rows: 11596
Number of fake rows: 611
'The Hard Times' scanning...


CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://thehardtimes.net/feeds


Number of articles found: 16


100%|██████████| 16/16 [00:03<00:00,  4.15it/s]


Number of added articles: 15 (We had: 0)
'The Shovel' scanning...


...

Number of articles found: 26


100%|██████████| 26/26 [00:10<00:00,  2.48it/s]
CRITICAL:newspaper.extractors:Must extract urls from either html, text or doc!
CRITICAL:newspaper.network:[REQUEST FAILED] HTTPSConnectionPool(host='www.burrardstreetjournal.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'www.burrardstreetjournal.com'. (_ssl.c:1010)")))
CRITICAL:newspaper.network:[REQUEST FAILED] HTTPSConnectionPool(host='www.burrardstreetjournal.com', port=443): Max retries exceeded with url: /feed (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'www.burrardstreetjournal.com'. (_ssl.c:1010)")))
CRITICAL:newspaper.network:[REQUEST FAILED] HTTPSConnectionPool(host='www.burrardstreetjournal.com', port=443): Max retries exceeded with url: /rss (Caused by SSLErr

Number of added articles: 10 (We had: 0)
'Burrard Street Journal' scanning...
Number of articles found: 0


0it [00:00, ?it/s]


Number of added articles: 0 (We had: 0)

SCRAPING AND COMBINING COMPLETED
Num of articles before: 11596
Num of added fake articles: 68
Current articles: 11664
Distributions:
label
0    10985
1      679
Name: count, dtype: int64
File saved: 'scraped_news_augmented.csv'


In [89]:
import requests
import time
import random
from bs4 import BeautifulSoup


We use different user agents to look like a real browser, not a bot.

In [90]:
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36'
]

In [91]:
def get_links_from_homepage(url):
    """
    Visits the homepage acting like a human user to collect article links.
    We use this because the standard 'newspaper.build' method was getting blocked.
    """
    try:
        # Select a random user agent
        headers = {'User-Agent': random.choice(user_agents)}

        # Send request with timeout
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200: return []

        soup = BeautifulSoup(response.content, 'html.parser')
        links = set()

        # Find all 'a' tags (links)
        for a_tag in soup.find_all('a', href=True):
            link = a_tag['href']

            # If link is relative (e.g., /article/123), add the base url
            if link.startswith('/'):
                link = url.rstrip('/') + link

            # Basic heuristic to filter only article links (length check + url match)
            if len(link) > 30 and url in link:
                links.add(link)

        return list(links)
    except:
        return []

In [92]:
def download_article_safe(url):
    """
    Downloads the article content using requests (to bypass 403)
    and then parses it with newspaper3k.
    """
    try:
        headers = {'User-Agent': random.choice(user_agents)}
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            article = Article(url)
            article.download(input_html=response.content) # Feed the HTML we downloaded manually
            article.parse()
            return article.title, article.text
        return None, None
    except:
        return None, None

In [93]:
current_filename = "scraped_news_augmented.csv"

try:
    # Load existing data to avoid duplicates
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())
        print(f"Current Data Loaded: {len(df_main)} rows")
        print(f"Current Fake News Count: {len(df_main[df_main['label']==1])}")
    else:
        print("File not found, creating new dataframe.")
        df_main = pd.DataFrame()
        existing_urls = set()
except:
    df_main = pd.DataFrame()
    existing_urls = set()

Current Data Loaded: 11664 rows
Current Fake News Count: 679


In [94]:
targets = [
    {"url": "https://www.theonion.com", "label": 1, "source": "The Onion"},
    {"url": "https://babylonbee.com", "label": 1, "source": "The Babylon Bee"},
    {"url": "https://www.duffelblog.com", "label": 1, "source": "Duffel Blog"},
    {"url": "https://reductress.com", "label": 1, "source": "Reductress"},
    {"url": "https://www.clickhole.com", "label": 1, "source": "ClickHole"},
    {"url": "https://waterfordwhispersnews.com", "label": 1, "source": "Waterford Whispers"}
]

In [95]:
new_fake_data = []

In [96]:
for target in targets:
    print(f"\n--- Scanning '{target['source']}' (Advanced Mode) ---")

    # 1. Collect Links from Homepage
    links = get_links_from_homepage(target['url'])
    print(f"Potential links found: {len(links)}")

    # 2. Scrape Content
    count = 0
    # Limit to 50 articles per site to be fast
    for link in tqdm(links[:50]):
        if link in existing_urls: continue # Skip if we already have it

        title, text = download_article_safe(link)

        # Check if text is valid and long enough
        if text and len(text) > 150:
            new_fake_data.append({
                "title": title,
                "news_url": link,
                "source": target['source'],
                "label": target['label'],
                "full_text": text
            })
            existing_urls.add(link)
            count += 1

            # Sleep to be polite and avoid ban
            time.sleep(random.uniform(0.5, 1.5))

    print(f"Added: {count} articles")

# SAVING
if len(new_fake_data) > 0:
    df_new = pd.DataFrame(new_fake_data)

    # Combining old and new data
    df_final = pd.concat([df_main, df_new], ignore_index=True)

    # Shuffling
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    # Save to csv
    df_final.to_csv(current_filename, index=False)
    print(f"\nSUCCESS! New Total Fake News: {len(df_final[df_final['label']==1])}")
else:
    print("\nNo new data retrieved. Sites are strictly protected.")


--- Scanning 'The Onion' (Advanced Mode) ---
Potential links found: 7


100%|██████████| 7/7 [00:07<00:00,  1.02s/it]


Added: 5 articles

--- Scanning 'The Babylon Bee' (Advanced Mode) ---
Potential links found: 49


100%|██████████| 49/49 [00:20<00:00,  2.39it/s]


Added: 13 articles

--- Scanning 'Duffel Blog' (Advanced Mode) ---
Potential links found: 28


...

Added: 12 articles

--- Scanning 'ClickHole' (Advanced Mode) ---
Potential links found: 2


100%|██████████| 2/2 [00:01<00:00,  1.20it/s]


Added: 2 articles

--- Scanning 'Waterford Whispers' (Advanced Mode) ---
Potential links found: 126


100%|██████████| 50/50 [01:01<00:00,  1.24s/it]


Added: 44 articles

SUCCESS! New Total Fake News: 764


It does not budge, we will try a different way for scraping.

In [98]:
current_filename = "scraped_news_augmented.csv"

try:
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())
        print(f"Current Data Loaded: {len(df_main)} rows")
        print(f"Current Fake News Count: {len(df_main[df_main['label']==1])}")
    else:
        print(f"File '{current_filename}' not found. Starting fresh.")
        df_main = pd.DataFrame()
        existing_urls = set()
except:
    df_main = pd.DataFrame()
    existing_urls = set()

# 2. DEEP TARGET LIST (Specific Categories & Archives)
# We target specific sections to find older articles
targets = [
    # The Onion Categories
    {"url": "https://www.theonion.com/latest", "label": 1, "source": "The Onion"},
    {"url": "https://www.theonion.com/politics", "label": 1, "source": "The Onion"},
    {"url": "https://www.theonion.com/local", "label": 1, "source": "The Onion"},

    # Babylon Bee Categories
    {"url": "https://babylonbee.com/news", "label": 1, "source": "Babylon Bee"},
    {"url": "https://babylonbee.com/politics", "label": 1, "source": "Babylon Bee"},

    # Reductress Archive
    {"url": "https://reductress.com/news/", "label": 1, "source": "Reductress"},

    # The Hard Times Categories
    {"url": "https://thehardtimes.net/news/", "label": 1, "source": "The Hard Times"},
    {"url": "https://thehardtimes.net/music/", "label": 1, "source": "The Hard Times"},

    # Duffel Blog Archive
    {"url": "https://www.duffelblog.com/archive", "label": 1, "source": "Duffel Blog"},

    # The Beaverton News
    {"url": "https://www.thebeaverton.com/news/", "label": 1, "source": "The Beaverton"},

    # Waterford Whispers News
    {"url": "https://waterfordwhispersnews.com/category/news/", "label": 1, "source": "Waterford Whispers"},

    # The Chaser News
    {"url": "https://chaser.com.au/news/", "label": 1, "source": "The Chaser"}
]

new_fake_data = []

# 3. SCRAPING LOOP
for target in targets:
    print(f"\n--- Scanning: {target['url']} ---")

    try:
        # memoize_articles=False ensures we re-scan for new links
        paper = newspaper.build(target['url'], memoize_articles=False)
        print(f"Links found: {len(paper.articles)}")

        # Try to get up to 300 articles per category
        limit = 300
        count = 0

        for article in tqdm(paper.articles[:limit]):
            # Check if URL already exists in our dataset
            if article.url in existing_urls:
                continue

            try:
                article.download()
                article.parse()

                # Filter: Ignore very short texts (likely errors)
                if len(article.text) > 150:
                    new_fake_data.append({
                        "title": article.title,
                        "news_url": article.url,
                        "source": target['source'],
                        "label": target['label'],
                        "full_text": article.text
                    })

                    existing_urls.add(article.url)
                    count += 1
            except:
                continue

        print(f"Added: {count}")

    except Exception as e:
        print(f"Error: {e}")

# 4. MERGE AND SAVE
if len(new_fake_data) > 0:
    df_new = pd.DataFrame(new_fake_data)

    # Concatenate old and new data
    df_augmented = pd.concat([df_main, df_new], ignore_index=True)

    # Shuffle the dataset
    df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\n--- FINAL REPORT ---")
    print(f"Previous Total: {len(df_main)}")
    print(f"Newly Added Fake News: {len(df_new)}")
    print(f"NEW TOTAL: {len(df_augmented)}")
    print(f"New Distribution:\n{df_augmented['label'].value_counts()}")

    # Overwrite the file
    df_augmented.to_csv(current_filename, index=False)
    print(f"File updated: '{current_filename}'")

    # Update df for further analysis
    df = df_augmented
else:
    print("\nNo new data found.")

Current Data Loaded: 11749 rows
Current Fake News Count: 764

--- Scanning: https://www.theonion.com/latest ---


CRITICAL:newspaper.network:[REQUEST FAILED] 403 Client Error: Forbidden for url: https://membership.theonion.com/
CRITICAL:newspaper.network:[REQUEST FAILED] 403 Client Error: Forbidden for url: https://membership.theonion.com/
CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://theonion.com/feeds


Links found: 150


100%|██████████| 150/150 [00:12<00:00, 11.77it/s]


Added: 3

--- Scanning: https://www.theonion.com/politics ---


...

Added: 58

--- Scanning: https://chaser.com.au/news/ ---


CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://chaser.com.au/feeds


Links found: 16


100%|██████████| 16/16 [00:02<00:00,  5.87it/s]


Added: 2

--- FINAL REPORT ---
Previous Total: 11749
Newly Added Fake News: 340
NEW TOTAL: 12089
New Distribution:
label
0    10985
1     1104
Name: count, dtype: int64
File updated: 'scraped_news_augmented.csv'


In [100]:
current_filename = "scraped_news_augmented.csv"

try:
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())
        print(f"Current Data Count: {len(df_main)}")
        print(f"Current Fake News Count: {len(df_main[df_main['label']==1])}")
    else:
        print("File not found!")
        df_main = pd.DataFrame()
        existing_urls = set()
except:
    df_main = pd.DataFrame()
    existing_urls = set()

# 2. NEW "UNDERGROUND" TARGETS
# These sites are generally less protected and have large archives.
targets = [
    {"url": "https://genesiustimes.com/", "label": 1, "source": "Genesius Times"},
    {"url": "https://newsthump.com/", "label": 1, "source": "News Thump"},
    {"url": "https://www.dailysquat.com/", "label": 1, "source": "Daily Squat"},
    {"url": "https://thesleaze.co.uk/", "label": 1, "source": "The Sleaze"},
    {"url": "https://beaverton.com/", "label": 1, "source": "The Beaverton (Retry)"}
]

new_fake_data = []

# 3. SCRAPING LOOP
for target in targets:
    print(f"\n--- Scanning '{target['source']}' ---")

    try:
        # memoize=False: Force re-scan
        paper = newspaper.build(target['url'], memoize_articles=False)
        print(f"Links found: {len(paper.articles)}")

        # Set high limit to get everything available
        limit = 800
        count = 0
        skipped = 0

        for article in tqdm(paper.articles[:limit]):
            # Check if we already have this link
            if article.url in existing_urls:
                skipped += 1
                continue

            try:
                article.download()
                article.parse()

                # Filters: Ignore short texts
                if len(article.text) > 200:
                    new_fake_data.append({
                        "title": article.title,
                        "news_url": article.url,
                        "source": target['source'],
                        "label": target['label'],
                        "full_text": article.text
                    })
                    existing_urls.add(article.url)
                    count += 1
            except:
                continue

        print(f"Added: {count} (Skipped/Existing: {skipped})")

    except Exception as e:
        print(f"Site Error: {e}")

# 4. MERGE AND SAVE
if len(new_fake_data) > 0:
    df_new = pd.DataFrame(new_fake_data)

    # Merge
    df_final = pd.concat([df_main, df_new], ignore_index=True)

    # Shuffle
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\n--- FINAL RESULT ---")
    print(f"Previous Fake Count: {len(df_main[df_main['label']==1])}")
    print(f"Newly Added Fake: {len(df_new)}")
    print(f"TOTAL FAKE COUNT: {len(df_final[df_final['label']==1])}")

    # Overwrite
    df_final.to_csv(current_filename, index=False)
    print(f"File updated: '{current_filename}'")

    # Update variable
    df = df_final

else:
    print("\nNo new data found.")

Current Data Count: 12089
Current Fake News Count: 1104

--- Scanning 'Genesius Times' ---


CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://genesiustimes.com/feeds


Links found: 69


100%|██████████| 69/69 [00:32<00:00,  2.11it/s]


Added: 58 (Skipped/Existing: 0)

--- Scanning 'News Thump' ---


...

100%|██████████| 8/8 [00:30<00:00,  3.82s/it]


Added: 8 (Skipped/Existing: 0)

--- Scanning 'The Beaverton (Retry)' ---
Links found: 0


0it [00:00, ?it/s]

Added: 0 (Skipped/Existing: 0)

--- FINAL RESULT ---
Previous Fake Count: 1104
Newly Added Fake: 88
TOTAL FAKE COUNT: 1192


File updated: 'scraped_news_augmented.csv'


In [102]:
# 1. LOAD EXISTING DATA
current_filename = "scraped_news_augmented.csv"
output_filename = "scraped_news_augmented.csv"

# Anti-Blocking Headers
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36'
]

try:
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())
        print(f"Current Data Loaded: {len(df_main)} rows")
        print(f"Current Fake News Count: {len(df_main[df_main['label']==1])}")
    else:
        print("File not found. Starting fresh.")
        df_main = pd.DataFrame()
        existing_urls = set()
except:
    df_main = pd.DataFrame()
    existing_urls = set()

# 2. NEW TARGETS (FRESH SOURCES)
# Using RSS feeds where possible for better reliability
targets = [
    {"url": "https://dailycurrant.com/", "label": 1, "source": "The Daily Currant"},
    {"url": "http://www.newsbiscuit.com/", "label": 1, "source": "NewsBiscuit"},
    {"url": "https://breakingburgh.com/", "label": 1, "source": "Breakingburgh"},
    {"url": "https://thelapine.ca/", "label": 1, "source": "The Lapine"},
    {"url": "https://glossynews.com/", "label": 1, "source": "Glossy News"}
]

# Helper function to get links (tries RSS first, then HTML)
def get_links_smart(url):
    try:
        headers = {'User-Agent': random.choice(user_agents)}
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code != 200: return []

        soup = BeautifulSoup(response.content, 'html.parser')
        links = set()

        # Try to find article links
        for a_tag in soup.find_all('a', href=True):
            link = a_tag['href']
            # Fix relative links
            if link.startswith('/'):
                link = url.rstrip('/') + link

            # Filter for likely article links (length > 30)
            if len(link) > 30 and url.split('/')[2] in link:
                links.add(link)

        return list(links)
    except:
        return []

def download_article_safe(url):
    try:
        headers = {'User-Agent': random.choice(user_agents)}
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            article = Article(url)
            article.download(input_html=response.content)
            article.parse()
            return article.title, article.text
        return None, None
    except:
        return None, None

# 3. SCRAPING LOOP
new_fake_data = []

for target in targets:
    print(f"\n--- Scanning '{target['source']}' ---")

    # Get links
    links = get_links_smart(target['url'])
    print(f"Links found: {len(links)}")

    count = 0
    # Try to get up to 200 articles per site
    for link in tqdm(links[:200]):
        if link in existing_urls:
            continue

        title, text = download_article_safe(link)

        if text and len(text) > 200:
            new_fake_data.append({
                "title": title,
                "news_url": link,
                "source": target['source'],
                "label": target['label'],
                "full_text": text
            })
            existing_urls.add(link)
            count += 1

            # Sleep to avoid blocking
            time.sleep(random.uniform(0.5, 1.5))

    print(f"Added: {count}")

# 4. MERGE AND SAVE
if len(new_fake_data) > 0:
    df_new = pd.DataFrame(new_fake_data)

    # Merge with main dataframe
    df_final = pd.concat([df_main, df_new], ignore_index=True)

    # Shuffle
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\n--- FINAL STATUS ---")
    print(f"Previous Total: {len(df_main)}")
    print(f"Newly Added: {len(df_new)}")
    print(f"NEW TOTAL FAKE: {len(df_final[df_final['label']==1])}")

    # Save
    df_final.to_csv(output_filename, index=False)
    print(f"File updated: '{output_filename}'")

    df = df_final
else:
    print("\nNo new data retrieved.")

Current Data Loaded: 12177 rows
Current Fake News Count: 1192

--- Scanning 'The Daily Currant' ---
Links found: 27


100%|██████████| 27/27 [00:44<00:00,  1.64s/it]


Added: 27

--- Scanning 'NewsBiscuit' ---
Links found: 41


100%|██████████| 41/41 [01:05<00:00,  1.60s/it]


Added: 24

--- Scanning 'Breakingburgh' ---
Links found: 2


...

0it [00:00, ?it/s]


Added: 0

--- Scanning 'Glossy News' ---
Links found: 0


0it [00:00, ?it/s]

Added: 0

--- FINAL STATUS ---
Previous Total: 12177
Newly Added: 51


NEW TOTAL FAKE: 1243
File updated: 'scraped_news_augmented.csv'


It still does not budge

In [103]:
import requests

In [108]:
current_filename = "scraped_news_augmented.csv"

try:
    if os.path.exists(current_filename):
        df_main = pd.read_csv(current_filename)
        existing_urls = set(df_main['news_url'].unique())
        print(f"Current Data Count: {len(df_main)}")
        print(f"Current Fake News Count: {len(df_main[df_main['label']==1])}")
    else:
        print("File not found! Creating a new one.")
        df_main = pd.DataFrame()
        existing_urls = set()
except:
    df_main = pd.DataFrame()
    existing_urls = set()

# 2. TARGETS (Combined list for maximum reach)
targets = [
    {"url": "https://genesiustimes.com/", "label": 1, "source": "Genesius Times"},
    {"url": "https://newsthump.com/", "label": 1, "source": "News Thump"},
    {"url": "https://www.dailysquat.com/", "label": 1, "source": "Daily Squat"},
    {"url": "https://thesleaze.co.uk/", "label": 1, "source": "The Sleaze"},
    {"url": "https://beaverton.com/", "label": 1, "source": "The Beaverton (Retry)"}
]

new_fake_data = []

# 3. SCRAPING LOOP
for target in targets:
    print(f"\n--- Scanning '{target['source']}' ---")

    try:
        # memoize=False: Force re-scan
        paper = newspaper.build(target['url'], memoize_articles=False)
        print(f"Links found: {len(paper.articles)}")

        # Skip if no links found
        if len(paper.articles) == 0:
            print("No links found, skipping...")
            continue

        # Limit to 800 to try and get everything
        limit = 2000
        count = 0
        skipped = 0

        for article in tqdm(paper.articles[:limit]):
            # Check if we already have this link
            if article.url in existing_urls:
                skipped += 1
                continue

            try:
                article.download()
                article.parse()

                # Filters: Ignore short texts
                if len(article.text) > 200:
                    new_fake_data.append({
                        "title": article.title,
                        "news_url": article.url,
                        "source": target['source'],
                        "label": target['label'],
                        "full_text": article.text
                    })
                    existing_urls.add(article.url)
                    count += 1
            except:
                continue

        print(f"Added: {count} (Skipped/Existing: {skipped})")

    except Exception as e:
        print(f"Site Error: {e}")

# 4. MERGE AND SAVE (HANDLES BOTH CASES)
if len(new_fake_data) > 0:
    print(f"\n--- NEW DATA FOUND ---")
    df_new = pd.DataFrame(new_fake_data)

    # Merge old and new
    df_final = pd.concat([df_main, df_new], ignore_index=True)

    # Shuffle
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Previous Fake Count: {len(df_main[df_main['label']==1])}")
    print(f"Newly Added Fake: {len(df_new)}")
    print(f"TOTAL FAKE COUNT: {len(df_final[df_final['label']==1])}")

else:
    print(f"\n--- NO NEW DATA FOUND ---")
    print("Using existing data.")
    # If no new data, the final dataframe is just the main one
    df_final = df_main

# Save to CSV (Always save to ensure the file is ready for the next step)
df_final.to_csv(current_filename, index=False)
print(f"File saved/updated: '{current_filename}'")

# Update the global 'df' variable for visualization scripts
df = df_final
print("Ready for analysis.")

Current Data Count: 12228
Current Fake News Count: 1243

--- Scanning 'Genesius Times' ---


CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://genesiustimes.com/feeds


Links found: 69


100%|██████████| 69/69 [00:04<00:00, 14.34it/s]


Added: 0 (Skipped/Existing: 58)

--- Scanning 'News Thump' ---


...

CRITICAL:newspaper.network:[REQUEST FAILED] 404 Client Error: Not Found for url: https://thesleaze.co.uk/feeds


Links found: 8


100%|██████████| 8/8 [00:00<00:00, 68478.43it/s]

Added: 0 (Skipped/Existing: 8)

--- Scanning 'The Beaverton (Retry)' ---


Links found: 0
No links found, skipping...

--- NO NEW DATA FOUND ---
Using existing data.
File saved/updated: 'scraped_news_augmented.csv'
Ready for analysis.


Still not much,but thats something, finally we'll move on to visualization 2 . Unfortunately we will have to add more fake news later.